In [1]:
# The setup 

import sys
from datetime import date

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

sys.path.append("..")

from src.indexer import load_index

load_dotenv()

vectorstore = load_index()
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [13]:
# The prompt 

SYSTEM_PROMPT = """You are a virtual assistant answering questions from BNP Paribas individual customers in mainland France about fees, products and procedures.

Rules:
1. Base your answer ONLY on the passages below. Never use outside knowledge, even if you think you know the answer.
2. After each fact, cite the passage it comes from with its number in square brackets, like [1] or [2][3].
3. If the passages do not contain the answer, or only part of it, say so clearly and suggest contacting a BNP Paribas adviser. Never guess. Do not suggest an adviser when the passages fully answer the question.
4. Today's date is {today}. Some passages mention dates (end of a service, rules starting on a given date). Compare them with today's date, and never present a service as available if the passages say it has ended.
5. Answer in the language of the question. Copy amounts, deadlines and phone numbers exactly as they are written in the passages.
6. Answer only the question that was asked. Include the conditions and exceptions that change the answer (for example, a different deadline in some cases), but do not add information about other topics, even related ones. If the answer is negative (a service has ended, an operation is not possible) and the passages mention an alternative for the same need, give it in one sentence. Keep the answer short: a few sentences are usually enough.
7. Keep the strength of the original wording. If the passages say the customer must do something, do not present it as advice, and do not present advice as an obligation.

Passages:
{context}"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)

In [3]:
# Format the passages 

def format_passages(docs) -> str:
    blocks = []
    for i, doc in enumerate(docs, start=1):
        blocks.append(f"[{i}] (page {doc.metadata['page']})\n{doc.page_content}")
    return "\n\n".join(blocks)

In [14]:
# The chain 

llm = ChatAnthropic(model="claude-haiku-5-5")

chain = prompt | llm

In [9]:
# Ask a question 

def ask(question: str) -> None:
    docs = retriever.invoke(question)
    response = chain.invoke(
        {
            "question": question,
            "context": format_passages(docs),
            "today": date.today().isoformat(),
        }
    )
    print(response.text)
    print()
    print("Sources:")
    for i, doc in enumerate(docs, start=1):
        print(f"  [{i}] {doc.metadata['title']} ({doc.metadata['version']}), p.{doc.metadata['page']}")
    print()
    usage = response.usage_metadata
    print(f"Tokens: {usage['input_tokens']} in, {usage['output_tokens']} out")


ask("Quels sont les horaires d'ouverture de l'agence de Lyon Bellecour ?")

Les passages fournis ne contiennent pas les horaires de l'agence de Lyon Bellecour. Je ne peux donc pas vous les indiquer.

Ils mentionnent seulement les horaires du service réclamations par téléphone, au 3477, qui sont de 8 h à 20 h du lundi au vendredi et de 8 h à 18 h le samedi, hors jours fériés [4][5]. Ces horaires ne concernent pas l'agence elle-même.

Pour connaître les horaires de l'agence de Lyon Bellecour, contactez votre conseiller ou consultez le site mabanque.bnpparibas.

Sources:
  [1] Conditions et tarifs - Particuliers (Tarifs au 1er janvier 2026), p.5
  [2] Conditions et tarifs - Particuliers (Tarifs au 1er janvier 2026), p.2
  [3] Conditions et tarifs - Particuliers (Tarifs au 1er janvier 2026), p.26
  [4] Conditions et tarifs - Particuliers (Tarifs au 1er janvier 2026), p.32
  [5] Conditions et tarifs - Particuliers (Tarifs au 1er janvier 2026), p.6

Tokens: 2346 in, 388 out


In [15]:
TEST_QUESTIONS = [
    "Je peux encore utiliser Facilidom pour changer de banque ?",
    "J'ai combien de temps pour contester une opération que je n'ai pas faite ?",
    "How much does the Reflets option cost?",
    "Quels sont les horaires d'ouverture de l'agence de Lyon Bellecour ?",
]

for question in TEST_QUESTIONS:
    print("=" * 80)
    print(question)
    print("=" * 80)
    ask(question)
    print()

Je peux encore utiliser Facilidom pour changer de banque ?
Non, le service Facilidom n'est plus disponible : sa commercialisation a pris fin le 01/01/2026 et il a été supprimé définitivement le 30/04/2026 [1]. Pour changer de banque, vous pouvez utiliser le service Facilimouv', gratuit, qui transfère automatiquement vers votre nouveau compte les prélèvements et virements récurrents enregistrés sur un compte ouvert dans une autre banque au cours des 13 derniers mois [1].

Sources:
  [1] Conditions et tarifs - Particuliers (Tarifs au 1er janvier 2026), p.4
  [2] Conditions générales de la convention de compte Bienvenue (Édition avril 2026), p.10
  [3] Conditions générales de la convention de compte Bienvenue (Édition avril 2026), p.10
  [4] Conditions générales de la convention de compte Bienvenue (Édition avril 2026), p.10
  [5] Conditions et tarifs - Particuliers (Tarifs au 1er janvier 2026), p.20

Tokens: 2630 in, 394 out

J'ai combien de temps pour contester une opération que je n'ai

## Prompt log (tested on 4 questions: Facilidom, contester, Reflets in English, Lyon Bellecour)

| Version | Change | Result |
|---------|--------|--------|
| v1 | 6 rules: documents only, citations, I don't know, today's date, language and exact numbers, concision | All rules respected. Issues: scope creep, reflexive "contact an adviser", an obligation softened into advice |
| v2 | Rule 3: when NOT to suggest an adviser. Rule 6: precise scope. New rule 7: keep the strength of the wording | Scope creep and wording fixed. Regression: the Facilimouv' alternative disappeared |
| v3 | Rule 6: give the alternative when the answer is negative | Regression fixed, no new issue. Remaining: ambiguous Reflets price, to check at step 6 |

In [1]:
import sys

from dotenv import load_dotenv

sys.path.append("..")

from src.rag import RagAssistant

load_dotenv()

assistant = RagAssistant()

for question in [
    "Je peux encore utiliser Facilidom pour changer de banque ?",
    "Quels sont les horaires d'ouverture de l'agence de Lyon Bellecour ?",
]:
    result = assistant.answer(question)
    print("=" * 80)
    print(question)
    print("=" * 80)
    print(result.answer)
    print()
    print(f"Retrieved: {len(result.sources)} passages. Cited:")
    for number, doc in result.cited_sources:
        print(f"  [{number}] {doc.metadata['title']} ({doc.metadata['version']}), p.{doc.metadata['page']}")
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Je peux encore utiliser Facilidom pour changer de banque ?
Non, le service Facilidom n'est plus disponible. Sa commercialisation a pris fin le 01/01/2026 et le service a été supprimé définitivement le 30/04/2026 [1].

Pour le même besoin, le service Facilimouv', gratuit, permet de transférer automatiquement vers votre nouveau compte les prélèvements et virements récurrents enregistrés sur un compte ouvert dans une autre banque au cours des 13 derniers mois [1].

Retrieved: 5 passages. Cited:
  [1] Conditions et tarifs - Particuliers (Tarifs au 1er janvier 2026), p.4

Quels sont les horaires d'ouverture de l'agence de Lyon Bellecour ?
Les documents fournis ne contiennent pas les horaires de l'agence de Lyon Bellecour. Je vous invite à contacter votre conseiller BNP Paribas, qui pourra vous les communiquer. Le numéro non surtaxé de votre agence est disponible sur mabanque.bnpparibas [5].

Retrieved: 5 passages. Cited:
  [5] Conditions et tarifs - Particuliers (Tarifs au 1er janvier 2026)

In [1]:
import sys

from dotenv import load_dotenv

sys.path.append("..")

from src.rag import RagAssistant

load_dotenv()

assistant = RagAssistant()
result = assistant.answer(
    "Pour quelqu'un qui part trois fois par an aux États-Unis, la Visa Premier est rentabilisée par rapport à la Classic ?"
)
print(result.answer)
print()
for number, doc in result.cited_sources:
    print(f"  [{number}] {doc.metadata['title']}, p.{doc.metadata['page']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Les documents fournis ne permettent pas de répondre à cette question. Ils indiquent le coût mensuel des cartes, mais pas les avantages liés aux voyages, ni les frais appliqués aux paiements ou retraits aux États-Unis.

Sur le seul coût mensuel de la cotisation, dans le cadre de l'offre groupée Esprit Libre, pour les 25 ans et plus :
- Visa Classic : 8 € [2] (ou 7,03 € dans la formule A [3])
- Visa Premier : 16 € [2] (ou 7,96 € dans la formule A [3])

Le montant dépend donc de l'offre et de la formule souscrites. Les passages ne disent pas si une Visa Premier se rentabilise pour trois séjours par an aux États-Unis, et je ne peux pas le calculer à partir de ces seules informations. Décider si la carte vaut son coût relève d'une évaluation de votre situation personnelle : je vous invite à contacter un conseiller BNP Paribas, qui pourra étudier vos besoins de voyage et vos options.

  [2] Conditions et tarifs - Particuliers, p.12
  [3] Conditions et tarifs - Particuliers, p.14
